In [9]:
from datetime import date, timedelta

today = date.today()
window1_start = today - timedelta(days=7)
window1_end = today
window2_start = today - timedelta(days=14)
window2_end = today - timedelta(days=8)

date_windows = [
    (window1_start.isoformat(), window1_end.isoformat()),
    (window2_start.isoformat(), window2_end.isoformat()),
]

In [10]:
import requests

API_KEY = "gGscoHYJYRJ0dJdEebDQp6uHAqP3SQaaVixYeZhw"
BASE_URL = "https://api.nasa.gov/neo/rest/v1/feed"

all_records = []

for start_date, end_date in date_windows:
    params = {
        "start_date": start_date,
        "end_date": end_date,
        "api_key": API_KEY,
    }
    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
        payload = response.json()
    except requests.exceptions.RequestException as e:
        print(f"API call failed for {start_date} to {end_date}: {e}")
        continue

    # near_earth_objects is a dict keyed by date string — not a flat list
    for date_str, objects_on_date in payload["near_earth_objects"].items():
        all_records.extend(objects_on_date)

print(f"Pulled {len(all_records)} total records across {len(date_windows)} windows.")

Pulled 59 total records across 2 windows.


In [14]:
from pathlib import Path
neo_ids = [str(obj["neo_reference_id"]) for obj in all_records]
neo_ids = list(dict.fromkeys(neo_ids))

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/raw/extracted_ids.txt").write_text("\n".join(neo_ids), encoding="utf8")

from generate_sentinel_log import generate_sentinel_log
generate_sentinel_log(neo_ids)

[generate_sentinel_log] 59 input ids -> 59 log rows (6 dropped, 6 ghost ids injected) -> data\raw\ground_station_log.csv


WindowsPath('data/raw/ground_station_log.csv')

In [24]:
cleaned = []
for record in all_records:
    if record["close_approach_data"]:
        cleaned.append(record)

In [25]:
def safe_float(value, default=None):
    """Try to cast value to float; return default if it can't be converted."""
    try:
        return float(value)
    except (ValueError, TypeError):
        return default

In [28]:
velocity = safe_float(record["close_approach_data"][0]["relative_velocity"]["kilometers_per_hour"])